# EgoRecover-JEPA | Full EPIC-KITCHENS MP4 download to Google Drive

**Kaggle CPU / Internet ON · Six executable cells · No model training or GPU required**

Designed from the user's working Baseline 1 notebook. Connect your **existing** Kaggle secret `RCLONE_TOKEN` (OAuth token JSON) to this notebook. **Never paste it into a cell.** Video destination remains `gdrive:EgoRecover-JEPA/EPIC-KITCHENS/Pxx/videos/Pxx_XXX.MP4`. Existing V-JEPA feature shards and Baseline 1 runs are untouched.

**Important: V-JEPA 2 = 256px model input; V-JEPA 2.1 = 384px model input.** Both share the *same original MP4 files*. Decode/resize separately later in the model pipeline; this download notebook does **not** transcode, resize, synthesize blur, or change timestamps.

**Runbook:**
1. Enable **Internet** and use **CPU**; enable `RCLONE_TOKEN` in Kaggle Add-ons → Secrets.
2. Run **Cells 1–4**, then **Cell 5** (default `MODE='inventory'` means nothing downloads), then **Cell 6**. Review Drive inventory and quota.
3. To test downloads, edit Cell 1 to `MODE='pilot'`, rerun **Cells 1–6**. This transfers up to 3 available missing MP4s.
4. For bulk, edit Cell 1 to `MODE='bulk'` and rerun **Cells 1–6** each new Kaggle session. Run budgets and available disk are checked at each video. Stop and report failures rather than silently overwrite.
5. For separate Kaggle workers, configure `SHARD_COUNT` and a **unique `SHARD_ID`** on each notebook. Every notebook must have **authorized write access to the intended Drive destination**. Do not send your Drive OAuth token to teammates. Their individual OAuth access to a shared folder may produce differently owned files; confirm Drive ownership and storage accounting before large transfers.

This notebook enumerates video IDs from official action annotations and video-info metadata; some distributor metadata files may not be published in the current downloader repository. **Its manifest is only complete if the official video catalogue is available and matched**. Where a checksum or source size cannot be established, existing copies are flagged for review and not destroyed. Oversized recordings may be deferred until scratch capacity is adjusted.

References: [official EPIC downloader](https://github.com/epic-kitchens/epic-kitchens-download-scripts), [official annotations](https://github.com/epic-kitchens/epic-kitchens-100-annotations), [community partial mirror](https://huggingface.co/datasets/awsaf49/epic_kitchens_100). Follow the dataset license and applicable usage terms.

## Cell 1 — Environment and rclone (no secrets printed)
Kaggle only. `MODE='inventory'` is the safe default. Check available Google Drive space before switching to bulk. The shared rclone client may be throttled; a custom OAuth client must be paired with a token authorized for that client.

In [ ]:
# CELL 1 / 6 — Kaggle setup, user-owned Google Drive authentication and safety settings
import os, sys, re, json, time, csv, hashlib, shutil, subprocess, tempfile, math
from pathlib import Path

# Only raw MP4s are downloaded. Both V-JEPA 2 (256px) and 2.1 (384px)
# use this SAME original video collection; input resizing occurs during inference.
DRIVE_ROOT = 'gdrive:EgoRecover-JEPA'
DRIVE_VIDEOS = DRIVE_ROOT + '/EPIC-KITCHENS'  # same location as Baseline 1
DRIVE_REPORTS = DRIVE_ROOT + '/dataset_download'
OTHER_EXISTING_ROOTS = ['gdrive:EPIC-KITCHENS']  # scan, but never write here
HF_REPO = 'awsaf49/epic_kitchens_100'              # partial fast mirror

# Modes: 'inventory' (safe), 'pilot' (3 videos), 'bulk' (selected shard), 'verify'.
MODE = 'inventory'
SHARD_ID, SHARD_COUNT = 0, 1                      # teammate partition: e.g. 0/3,1/3,2/3
LEDGER_REMOTE = DRIVE_REPORTS + f'/shard_{SHARD_ID}_of_{SHARD_COUNT}/verified_ledger.csv'
INCLUDE_TEST_VIDEOS = True
PREFER_TRAINING_VIDEOS = True
PILOT_VIDEOS = 3
MAX_SESSION_HOURS = 8.0
MAX_SESSION_UPLOAD_GB = 125.0
MAX_STAGE_GB = 38.0
MIN_FREE_GB = 7.0
MAX_SINGLE_VIDEO_GB = 25.0
BRISTOL_RANGE_WORKERS = 4
FORCE_SOURCE = 'auto'  # 'auto', 'hf', 'bristol'; no fallback if source missing
# Never delete/replace existing Drive data automatically.

assert Path('/kaggle').exists(), 'This notebook is for Kaggle only (Internet ON, CPU accelerator).'
assert 0 <= SHARD_ID < SHARD_COUNT, 'Require 0 <= SHARD_ID < SHARD_COUNT.'
RUN_TAG = time.strftime('%Y%m%d_%H%M%S', time.gmtime())
WORK = Path('/kaggle/working/epic_full_reports')
STAGE = Path('/kaggle/temp/epic_full_staging')
WORK.mkdir(parents=True, exist_ok=True)
STAGE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(STAGE / 'hf_home')
os.environ['HF_XET_CACHE'] = str(STAGE / 'hf_xet')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'

p = subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'huggingface_hub', 'hf_xet', 'requests', 'pandas'], capture_output=True, text=True)
if p.returncode: raise RuntimeError('Python package installation failed: ' + p.stderr[-600:])
if not shutil.which('rclone'):
    p = subprocess.run(['bash', '-lc', 'curl -fsSL https://rclone.org/install.sh | bash'], capture_output=True, text=True)
    if p.returncode: raise RuntimeError('rclone installation failed: ' + p.stderr[-600:])

def rc(args, timeout=600, allow_error=False):
    p = subprocess.run(['rclone', *args], capture_output=True, text=True, timeout=timeout)
    if p.returncode and not allow_error:
        raise RuntimeError('rclone failed (' + ' '.join(args[:2]) + '): ' + p.stderr[-500:])
    return p

from kaggle_secrets import UserSecretsClient
secret = UserSecretsClient().get_secret('RCLONE_TOKEN')
assert secret, 'Enable your existing RCLONE_TOKEN secret for this Kaggle notebook.'
try:
    token = json.loads(secret)
    assert isinstance(token, dict) and 'refresh_token' in token
except Exception as e:
    raise RuntimeError('RCLONE_TOKEN must contain the JSON OAuth token from your original Baseline 1 notebook.') from e

cfgdir = Path.home()/'.config'/'rclone'
cfgdir.mkdir(parents=True, exist_ok=True)
conf = cfgdir/'rclone.conf'
# Optional separate OAuth client secrets can mitigate throttling. Never print their values.
try: client_id = UserSecretsClient().get_secret('RCLONE_CLIENT_ID') or ''
except Exception: client_id = ''
try: client_secret = UserSecretsClient().get_secret('RCLONE_CLIENT_SECRET') or ''
except Exception: client_secret = ''
lines = ['[gdrive]', 'type = drive', 'scope = drive', 'token = ' + json.dumps(token, separators=(',', ':'))]
if client_id and client_secret:
    lines += ['client_id = ' + client_id, 'client_secret = ' + client_secret]
conf.write_text('\n'.join(lines) + '\n')
conf.chmod(0o600)
del secret, token, lines

assert rc(['lsf','gdrive:','--max-depth','1'], timeout=120).returncode == 0
rc(['mkdir', DRIVE_VIDEOS]); rc(['mkdir', DRIVE_REPORTS])
print('Connected to Google Drive via RCLONE_TOKEN (credentials hidden).')
if not client_id: print('NOTE: Your shared rclone OAuth client may be throttled; use a private client for high-volume uploads if possible.')
print('Drive destination:', DRIVE_VIDEOS)
print('Kaggle free staging GiB:', round(shutil.disk_usage(STAGE).free/2**30, 2))
about = rc(['about','gdrive:'], timeout=120, allow_error=True)
if about.returncode == 0: print('Google Drive quota:\n', about.stdout)
else: print('Drive quota not provided by remote; verify capacity in Google Drive settings before bulk mode.')


## Cell 2 — Official dataset manifest
This is a **raw MP4** inventory, not an action-training split. It includes unlabeled/test recordings when official metadata exposes them. V-JEPA training, evaluation and historic `val_heldout` assignments are **not** modified here.

In [ ]:
# CELL 2 / 6 — Official annotation inventory + optional source checksums + Hugging Face file listing
import pandas as pd
import requests
from huggingface_hub import HfApi
from huggingface_hub.hf_api import RepoFile

ANNO = STAGE/'official_annotations'
SCRIPTS = STAGE/'official_download_scripts'
for url, path in [('https://github.com/epic-kitchens/epic-kitchens-100-annotations', ANNO),
                  ('https://github.com/epic-kitchens/epic-kitchens-download-scripts', SCRIPTS)]:
    if not path.is_dir():
        p = subprocess.run(['git','clone','-q','--depth','1',url,str(path)], capture_output=True,text=True)
        if p.returncode: raise RuntimeError('Cannot fetch required official repository: '+url+' '+p.stderr[-350:])

video_splits = {}
for split, filename, required in [('train','EPIC_100_train.csv',True),('val','EPIC_100_validation.csv',True),
                                  ('test','EPIC_100_test_timestamps.csv',False)]:
    if split == 'test' and not INCLUDE_TEST_VIDEOS: continue
    file = ANNO/filename
    if not file.exists():
        if required: raise FileNotFoundError(str(file))
        print('Optional test timestamps unavailable:', filename)
        continue
    df = pd.read_csv(file)
    if 'video_id' not in df:
        if required: raise ValueError(filename + ' has no video_id field')
        print('Optional test metadata does not expose video_id:', filename)
        continue
    for vid in df.video_id.dropna().astype(str).str.upper().unique():
        if re.fullmatch(r'P\d\d_\d{2,3}',vid):
            video_splits.setdefault(vid,set()).add(split)

# EPIC_100_video_info.csv helps include the complete recorded-video catalogue,
# not only videos with action annotations (important for a *full* raw video archive).
video_info = ANNO/'EPIC_100_video_info.csv'
if video_info.exists():
    vi=pd.read_csv(video_info)
    if 'video_id' in vi:
        for vid in vi.video_id.dropna().astype(str).str.upper().unique():
            if re.fullmatch(r'P\d\d_\d{2,3}',vid):video_splits.setdefault(vid,set()).add('catalogue')
else:
    print('WARNING: complete video_info catalogue unavailable. Do not claim full dataset coverage.')

# The downloader's full challenge-based split list can contain videos absent from annotated train/val.
extra_split = SCRIPTS/'data'/'epic_100_splits.csv'
if extra_split.exists():
    df = pd.read_csv(extra_split)
    for vid in df.get('video_id',pd.Series(dtype=str)).dropna().astype(str).str.upper().unique():
        if re.fullmatch(r'P\d\d_\d{2,3}',vid): video_splits.setdefault(vid,set()).add('official_additional')
else:
    print('NOTE: epic_100_splits.csv absent from current downloader repo; inventory is based on available official annotations.')

s55 = {}
f55 = SCRIPTS/'data'/'epic_55_splits.csv'
if f55.exists():
    x = pd.read_csv(f55)
    s55 = dict(zip(x.video_id.astype(str), x['split'].astype(str)))
else:
    print('NOTE: epic_55_splits.csv absent; original EPIC-55 videos will probe documented train/test Bristol paths.')

md5s = {}
md5file = SCRIPTS/'data'/'md5.csv'
if md5file.exists():
    x = pd.read_csv(md5file)
    for row in x.itertuples(index=False):
        md5s[(str(row.version),str(row.file_remote_path))] = str(row.md5).lower()
else:
    print('NOTE: md5.csv absent; checksums from the original download-script metadata will not be available.')

hf_files = {}
api = HfApi()
try:
    for obj in api.list_repo_tree(repo_id=HF_REPO, repo_type='dataset', recursive=True):
        if not isinstance(obj, RepoFile) or not obj.path.lower().endswith('.mp4'): continue
        vid = Path(obj.path).stem.upper()
        if not re.fullmatch(r'P\d\d_\d{2,3}',vid): continue
        hf_files[vid] = dict(path=obj.path, size=int(obj.size or 0))
except Exception as e:
    print('HF inventory unavailable; will use Bristol where possible:',type(e).__name__,str(e)[:180])

BASE55='https://data.bris.ac.uk/datasets/3h91syskeag572hl6tvuovwv4d'
BASE100='https://data.bris.ac.uk/datasets/2g1n6qdydwa9u22shpxqzp0t8m'
rows=[]
for vid,splits in sorted(video_splits.items()):
    pid=vid.split('_')[0]
    extension=len(vid.split('_')[1])==3
    official_paths = ([f'{pid}/videos/{vid}.MP4'] if extension else
                      [f'videos/{q}/{pid}/{vid}.MP4' for q in ([s55[vid]] if vid in s55 else ['train','test'])])
    version='100' if extension else '55'
    checks=[md5s.get((version,path),'') for path in official_paths]
    official_md5 = next((c for c in checks if re.fullmatch('[a-f0-9]{32}',c)), '')
    remote_urls = [(BASE100 if extension else BASE55)+'/'+p for p in official_paths]
    hf = hf_files.get(vid,{})
    primary = 'hf' if hf else 'bristol'
    rows.append(dict(video_id=vid, official_split='|'.join(sorted(splits)), participant=pid,
                     extension=extension, source=primary, hf_path=hf.get('path',''), hf_size=int(hf.get('size',0)),
                     official_md5=official_md5, bristol_urls=json.dumps(remote_urls),
                     existing_path='', existing_size=0, existing_status='missing', status='pending', last_error=''))

manifest = pd.DataFrame(rows)
if manifest.empty: raise RuntimeError('No official video IDs discovered: stop and inspect annotation repositories.')
manifest.to_csv(WORK/'manifest_sources.csv',index=False)
print('Official video IDs:',len(manifest),'| HF available:',sum(manifest.source=='hf'))
print('Official split distribution:',manifest.official_split.value_counts().head(8).to_dict())
print('HF published file sizes (GB, partial mirror):',round(manifest.hf_size.sum()/1e9,1))
print('Available official hashes:',sum(manifest.official_md5.astype(bool)))
print('Metadata status: sources enumerated; Drive inventory is next.')


## Cell 3 — Inventory existing Drive MP4s
Scans existing paths. Never overwrites or automatically repairs a file merely because its filename matches. Hash verified = strongest. Matching source size alone is a weaker check.

In [ ]:
# CELL 3 / 6 — Remote inventory, size/hash checks, skip existing videos (no overwrite)

def list_drive_mp4s(root):
    proc = rc(['lsjson',root,'--recursive','--files-only','--hash'], timeout=1200, allow_error=True)
    if proc.returncode:
        # Existing alternative root may not exist. Canonical root must work.
        if root == DRIVE_VIDEOS: raise RuntimeError('Cannot inventory destination: '+proc.stderr[-300:])
        print('Optional inventory root not accessible:',root)
        return {}
    files = {}
    for entry in json.loads(proc.stdout or '[]'):
        if entry.get('IsDir') or not entry.get('Path','').lower().endswith('.mp4'): continue
        vid = Path(entry['Path']).stem.upper()
        if not re.fullmatch(r'P\d\d_\d{2,3}',vid):continue
        files.setdefault(vid,[]).append(dict(remote=root.rstrip('/')+'/'+entry['Path'],size=int(entry.get('Size',0)),
                        md5=(entry.get('Hashes') or {}).get('MD5','').lower()))
    return files

# Load small persistent, verified-upload ledger from Drive. This is essential for
# original Bristol MP4s where the official MD5 manifest is missing. Remote hash
# and size are compared again on every restart (never trust the ledger alone).
from io import StringIO
verified_uploads={}
ledger_result=rc(['cat',LEDGER_REMOTE],allow_error=True,timeout=180)
if ledger_result.returncode==0 and ledger_result.stdout.strip():
    old=pd.read_csv(StringIO(ledger_result.stdout)).fillna('')
    for record in old.to_dict('records'):
        try:
            v=str(record['video_id']).upper()
            verified_uploads[v]=dict(video_id=v,md5=str(record['md5']).lower(),size=int(record['size']),
                                     remote=str(record['remote']))
        except Exception: pass
print('Previously verified upload ledger:',len(verified_uploads),'videos')

existing={}
for root in [DRIVE_VIDEOS]+OTHER_EXISTING_ROOTS:
    found=list_drive_mp4s(root)
    for vid,files in found.items(): existing.setdefault(vid,[]).extend(files)
    print('Scanned',root,':',sum(map(len,found.values())),'MP4 files')

for i,row in manifest.iterrows():
    candidates = existing.get(row.video_id,[])
    if not candidates:continue
    status='existing_unverified'; chosen=None
    for item in candidates:
        expected=int(row.hf_size)
        known_md5=str(row.official_md5 or '')
        ledger=verified_uploads.get(row.video_id)
        if (ledger and item['md5'] and item['md5']==ledger['md5'] and
            item['size']==ledger['size'] and item['remote']==ledger['remote']):
            status='existing_hash_verified';chosen=item;break
        if known_md5 and item['md5'] == known_md5:
            status='existing_hash_verified';chosen=item;break
        if expected and item['size']==expected and item['size']>0:
            status='existing_size_verified';chosen=item;break
    if chosen is None:
        chosen=max(candidates,key=lambda x:x['size'])
    manifest.at[i,'existing_path']=chosen['remote']
    manifest.at[i,'existing_size']=chosen['size']
    manifest.at[i,'existing_status']=status
    # Never overwrite or redownload an existing path until the user audits it.
    manifest.at[i,'status']=status

# Preserve records of the old 69-video baseline; no training or encoded-feature changes.
manifest.to_csv(WORK/'master_manifest.csv',index=False)
print('\nInventory:',manifest.status.value_counts().to_dict())
print('Missing original MP4s:',sum(manifest.status=='pending'))
print('Existing but not verified:',sum(manifest.status=='existing_unverified'))
print('Known missing HF volume (GB):',round(manifest.loc[(manifest.status=='pending') & (manifest.hf_size>0),'hf_size'].sum()/1e9,1))
print('Drive transfer target remains:',DRIVE_VIDEOS)


## Cell 4 — Transfer engine
Downloads to Kaggle scratch and uploads via rclone, checks local MD5 against the Google Drive reported MD5, then deletes only the *verified* local copy. For Bristol, HTTP Range connections are used when available. A failed download/upload is logged and preserves existing Drive data.

In [ ]:
# CELL 4 / 6 — Safe MP4 download and verified upload helpers
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import hf_hub_download
import requests

GB=1_000_000_000
SESSION_LOG=WORK/'session_log.csv'
log_rows=[]

def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def verified_remote(vid, expected_size, expected_md5=''):
    root=DRIVE_VIDEOS+'/'+vid[:3]+'/videos'
    p=rc(['lsjson',root,'--files-only','--hash'],allow_error=True,timeout=240)
    if p.returncode:return None
    for item in json.loads(p.stdout or '[]'):
        if item.get('Name','').upper()!=vid.upper()+'.MP4': continue
        if int(item.get('Size',0))!=int(expected_size):return None
        actual=(item.get('Hashes') or {}).get('MD5','').lower()
        if expected_md5 and actual!=expected_md5:return None
        return root+'/'+item['Name']
    return None

def available_stage_bytes():
    # 1 file at a time; limit size and maintain reserved disk space.
    return min(int(MAX_STAGE_GB*GB),shutil.disk_usage(STAGE).free-int(MIN_FREE_GB*GB))

def bristol_head(url):
    # Tiny range probe; do not download a complete large file to inspect its size.
    with requests.get(url,headers={'Range':'bytes=0-0','Accept-Encoding':'identity'},
                      timeout=(20,40),stream=True,allow_redirects=True) as r:
        if r.status_code==206:
            m=re.match(r'bytes\s+0-0/(\d+)',r.headers.get('Content-Range',''))
            if m:return int(m.group(1)),True
        if r.status_code==200:
            n=r.headers.get('Content-Length')
            if n and n.isdigit():return int(n),False
        raise RuntimeError('Source range probe HTTP '+str(r.status_code)+' or size unavailable')

def ranged_download(url, target, size, workers=4):
    # Allocate and fill nonoverlapping byte ranges, validating each Content-Range.
    # In-progress bytes live on Kaggle scratch; verified MP4s live on Drive.
    part=Path(str(target)+'.part')
    part.parent.mkdir(parents=True,exist_ok=True)
    part.unlink(missing_ok=True) # never trust an incomplete file without a range ledger
    chunk_size=max(8*1024*1024,math.ceil(size/workers))
    ranges=[(a,min(a+chunk_size-1,size-1)) for a in range(0,size,chunk_size)]
    with open(part,'wb') as f:f.truncate(size)
    def grab(bounds):
        start,end=bounds
        hdr={'Range':f'bytes={start}-{end}','Accept-Encoding':'identity'}
        with requests.get(url,headers=hdr,stream=True,timeout=(30,180)) as r:
            if r.status_code!=206:raise RuntimeError('Range request not honored, HTTP '+str(r.status_code))
            m=re.match(r'bytes\s+(\d+)-(\d+)/(\d+)',r.headers.get('Content-Range',''))
            if not m or tuple(map(int,m.groups()))!=(start,end,size):raise RuntimeError('Incorrect Content-Range')
            offset=start
            fd=os.open(part,os.O_WRONLY)
            try:
                for b in r.iter_content(chunk_size=4*1024*1024):
                    if not b:continue
                    if offset+len(b)>end+1:raise RuntimeError('Overlong HTTP range')
                    n=os.pwrite(fd,b,offset)
                    if n!=len(b):raise IOError('Incomplete local pwrite')
                    offset+=len(b)
            finally:os.close(fd)
            if offset!=end+1:raise RuntimeError('Truncated HTTP range')
    try:
        with ThreadPoolExecutor(max_workers=min(workers,len(ranges))) as ex:
            for future in as_completed([ex.submit(grab,r) for r in ranges]):future.result()
        if part.stat().st_size!=size:raise RuntimeError('Local size mismatch')
        part.replace(target)
        return target
    except Exception:
        part.unlink(missing_ok=True) # never mistake a partially allocated sparse file for a valid MP4
        raise

def bristol_download(row, local):
    problems=[]
    for url in json.loads(row.bristol_urls):
        try:
            size,range_ok=bristol_head(url)
            if size>available_stage_bytes():raise RuntimeError('Video exceeds available staging limit')
            if range_ok:
                return ranged_download(url,local,size,BRISTOL_RANGE_WORKERS)
            # No range support; single streaming transfer with strict byte limit.
            capacity=available_stage_bytes()
            with requests.get(url,stream=True,timeout=(30,180)) as resp:
                resp.raise_for_status()
                actual=0
                with open(str(local)+'.part','wb') as f:
                    for block in resp.iter_content(4*1024*1024):
                        if block:
                            actual+=len(block)
                            if actual>size or actual>capacity:raise RuntimeError('Staging limit exceeded')
                            f.write(block)
            if actual!=size:raise RuntimeError('Truncated response')
            Path(str(local)+'.part').replace(local)
            return local
        except Exception as e:
            Path(str(local)+'.part').unlink(missing_ok=True)
            local.unlink(missing_ok=True)
            problems.append(type(e).__name__+': '+str(e)[:150])
    raise RuntimeError('All official URLs failed: '+'; '.join(problems))

def fetch_video(row):
    vid=row.video_id
    directory=STAGE/'videos'/vid[:3]
    directory.mkdir(parents=True,exist_ok=True)
    local=directory/(vid+'.MP4')
    if row.source=='hf' and FORCE_SOURCE!='bristol':
        size=int(row.hf_size)
        if not size or size>available_stage_bytes():raise RuntimeError('HF file too large or size unknown for staging')
        # local_dir makes one local copy; avoid separate full-sized Hub cache blobs.
        local=Path(hf_hub_download(repo_id=HF_REPO,repo_type='dataset',filename=row.hf_path,
                 local_dir=str(STAGE/'hf_downloads')))
        if local.stat().st_size!=size:raise RuntimeError('HF file size mismatch')
    else:
        if FORCE_SOURCE=='hf': raise RuntimeError('HF source unavailable for '+vid)
        local=bristol_download(row,local)
    if row.official_md5:
        check=md5_file(local)
        if check.lower()!=str(row.official_md5).lower():
            raise RuntimeError('Downloaded MD5 differs from official checksum; file kept locally for investigation')
    return local

def upload_one(row,local):
    vid=row.video_id; target=DRIVE_VIDEOS+'/'+vid[:3]+'/videos/'+vid+'.MP4'
    size=local.stat().st_size
    checksum=md5_file(local)
    # Safety: never overwrite any existing destination path (even when corrupted).
    if verified_remote(vid,size,checksum):
        verified_uploads[vid]=dict(video_id=vid,size=size,md5=checksum,remote=target)
        local.unlink(missing_ok=True)
        return 'already_verified',target
    parent=target.rsplit('/',1)[0]
    p=rc(['lsf',parent,'--files-only'],timeout=120,allow_error=True)
    if p.returncode==0 and (vid+'.MP4') in p.stdout.splitlines():
        raise RuntimeError('Destination exists but differs; refusing overwrite: '+target)
    rc(['mkdir',parent])
    rc(['copyto',str(local),target,'--ignore-existing','--drive-chunk-size','64M',
        '--retries','3','--low-level-retries','8','--stats','60s'],timeout=3*3600)
    verified=verified_remote(vid,size,checksum)
    if not verified:raise RuntimeError('Drive MD5/size verification failed; local copy retained: '+target)
    # The uploaded file has matching checksum and size: safe to release scratch copy.
    verified_uploads[vid]=dict(video_id=vid,size=size,md5=checksum,remote=target)
    local.unlink(missing_ok=True)
    return 'uploaded_hash_verified',target

def report_snapshot():
    manifest.to_csv(WORK/'master_manifest.csv',index=False)
    pd.DataFrame(log_rows,columns=['time','video_id','status','gb','seconds','detail']).to_csv(SESSION_LOG,index=False)
    summary=dict(updated_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
                 videos=len(manifest),status_counts=manifest.status.value_counts().to_dict(),
                 shard_id=SHARD_ID,shard_count=SHARD_COUNT,output=DRIVE_VIDEOS)
    (WORK/'summary.json').write_text(json.dumps(summary,indent=2))
    ledger_local=WORK/'verified_ledger.csv'
    pd.DataFrame(list(verified_uploads.values()),columns=['video_id','size','md5','remote']).to_csv(ledger_local,index=False)
    # Maintain a separate ledger per deterministic teammate shard.
    ledger_upload=rc(['copyto',str(ledger_local),LEDGER_REMOTE],timeout=300,allow_error=True)
    if ledger_upload.returncode:print('WARNING: Ledger backup failed:',ledger_upload.stderr[-180:])
    p=rc(['copy',str(WORK),DRIVE_REPORTS+f'/shard_{SHARD_ID}_of_{SHARD_COUNT}/run_{RUN_TAG}',
          '--include','*.csv','--include','*.json'],timeout=600,allow_error=True)
    if p.returncode:print('WARNING: Report backup failed; Kaggle copies still exist:',p.stderr[-250:])

print('Transfer engine defined. No data downloaded yet.')


## Cell 5 — Pilot or bulk
**Default mode performs no MP4 transfers.** Set `MODE='pilot'` in Cell 1 for three small videos, then `MODE='bulk'` after checking speed and verification. Videos are partitioned by stable hashing if multiple sessions run simultaneously.

In [ ]:
# CELL 5 / 6 — Pilot or full-session transfer. Default MODE='inventory' means NO download.
# This cell is safe to rerun. Only a verified remote MP4 is considered complete.
if MODE not in ('inventory','pilot','bulk','verify'):
    raise ValueError('MODE must be inventory, pilot, bulk, or verify')

if MODE in ('inventory','verify'):
    print('MODE:',MODE,'— no video downloads or MP4 modifications.')
else:
    # Deterministic split among teammates: every shard gets distinct video IDs.
    queue = manifest[manifest.status.eq('pending')].copy()
    priority={'train':0,'val':1,'test':2}
    if PREFER_TRAINING_VIDEOS:
        queue['_order']=queue.official_split.apply(lambda s:min([priority.get(t,3) for t in str(s).split('|')]))
    else:queue['_order']=0
    queue['_source_order']=queue.source.map({'hf':0,'bristol':1}).fillna(2)
    queue=queue.sort_values(['_order','_source_order','hf_size','video_id'],kind='stable')
    queue=queue[queue.video_id.map(lambda s:int(hashlib.sha256(s.encode()).hexdigest()[:12],16)%SHARD_COUNT == SHARD_ID)]
    if MODE=='pilot':
        queue=queue[(queue.source=='hf') & (queue.hf_size>200_000_000) & (queue.hf_size<3_000_000_000)].head(PILOT_VIDEOS)
        print('Pilot chosen:',queue.video_id.tolist())
        if queue.empty:raise RuntimeError('No suitable HF pilot MP4 found; inspect sources in Cell 2.')

    uploaded_bytes=0; run_start=time.monotonic()
    for ix,row in queue.iterrows():
        runtime_h=(time.monotonic()-run_start)/3600
        if runtime_h >= MAX_SESSION_HOURS:
            print('Session time budget reached; new transfers stopped.');break
        if uploaded_bytes >= MAX_SESSION_UPLOAD_GB*GB:
            print('Session upload budget reached.');break
        if shutil.disk_usage(STAGE).free < (MIN_FREE_GB+2)*GB:
            print('Local free disk is low; stop without removing any failed file.');break
        if int(row.hf_size)>0 and int(row.hf_size)>MAX_SINGLE_VIDEO_GB*GB:
            print('Deferred oversized video:',row.video_id,round(row.hf_size/GB,1),'GB');continue
        t0=time.monotonic(); local=None
        print('\n[START]',row.video_id,'source:',row.source,flush=True)
        try:
            local=fetch_video(row)
            n=local.stat().st_size
            if n>MAX_SINGLE_VIDEO_GB*GB: raise RuntimeError('File exceeded per-video size safety limit')
            if uploaded_bytes+n>MAX_SESSION_UPLOAD_GB*GB:
                print('Session budget insufficient for this video; local file retained.');break
            result,path=upload_one(row,local)
            manifest.at[ix,'status']=result
            manifest.at[ix,'existing_path']=path
            manifest.at[ix,'existing_size']=n
            uploaded_bytes += (0 if result=='already_verified' else n)
            log_rows.append([time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),row.video_id,result,round(n/GB,3),round(time.monotonic()-t0,1),path])
            print('[OK]',row.video_id,result,round(n/GB,2),'GB',flush=True)
        except Exception as e:
            manifest.at[ix,'status']='failed_this_session'
            manifest.at[ix,'last_error']=(type(e).__name__+': '+str(e))[:350]
            log_rows.append([time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),row.video_id,'failed',0,round(time.monotonic()-t0,1),manifest.at[ix,'last_error']])
            print('[FAIL]',row.video_id,manifest.at[ix,'last_error'],flush=True)
            print('A failed local file is not deleted automatically; check staging disk before continuing.')
        finally:
            report_snapshot()
        print('Session uploaded:',round(uploaded_bytes/GB,2),'GB | elapsed:',round((time.monotonic()-run_start)/3600,2),'h')

    print('\nRUN SUMMARY: files uploaded this session:',sum(x[2]=='uploaded_hash_verified' for x in log_rows),
          '| GB:',round(uploaded_bytes/GB,2))


## Cell 6 — Verification summary and resumability
Reconciles canonical Drive storage and saves CSV/JSON reports. Re-run cells from the top after each Kaggle session. Incomplete or uncertain existing files remain visible for audit. Bulk downloads do not automatically train V-JEPA.

In [ ]:
# CELL 6 / 6 — Refresh Drive inventory, save report, identify remaining work
# No GPU / model loading / probe training / video deletion.
latest = list_drive_mp4s(DRIVE_VIDEOS)
completed=0
for ix,row in manifest.iterrows():
    candidates=latest.get(row.video_id,[])
    if not candidates:continue
    best=max(candidates,key=lambda v:v['size'])
    manifest.at[ix,'existing_path']=best['remote']
    manifest.at[ix,'existing_size']=best['size']
    expected_md5=str(row.official_md5 or '')
    known_hf_size=int(row.hf_size)
    ledger=verified_uploads.get(row.video_id)
    if ledger and best['md5'] and best['md5']==ledger['md5'] and best['size']==ledger['size'] and best['remote']==ledger['remote']:
        status='drive_hash_verified'
    elif expected_md5 and best['md5']==expected_md5:
        status='drive_hash_verified'
    elif known_hf_size and best['size']==known_hf_size:
        status='drive_size_verified'
    elif row.status in ('uploaded_hash_verified','already_verified'):
        status=row.status
    else:
        status='drive_unverified'
    manifest.at[ix,'status']=status
    if status in ('drive_hash_verified','drive_size_verified','uploaded_hash_verified','already_verified'):
        completed+=1

report_snapshot()
print('Total unique videos in manifest:',len(manifest))
print('Verified or source-size-checked on canonical Drive path:',completed)
print('Status counts:',manifest.status.value_counts().to_dict())
remaining=manifest[~manifest.status.isin(['drive_hash_verified','drive_size_verified','uploaded_hash_verified','already_verified'])]
remaining[['video_id','official_split','source','status','last_error']].to_csv(WORK/'remaining_or_unverified.csv',index=False)
report_snapshot()
print('Remaining / unverified:',len(remaining))
print('Logs and reports on Drive:',DRIVE_REPORTS)
print('Research compatibility: V-JEPA 2 uses 256px input; V-JEPA 2.1 uses 384px input; original MP4s unchanged.')
print('To continue on another Kaggle session: rerun Cells 1–4, set MODE=bulk in Cell 1, then Cell 5 and Cell 6.')
